# Curcumin Reference Docking

This notebook performs the same docking and scoring process for the natural ligand **Curcumin**.
This serves as a baseline (Ground Truth) to compare the performance of generated analogs.

In [1]:
from openbabel import pybel
from openbabel import openbabel as ob
from pathlib import Path
import sys
import pandas as pd
import os
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess
import glob
from datetime import datetime
from admet_ai import ADMETModel
import numpy as np
import shutil

## Configuration

Setup paths to the receptor and Curcumin source file.

In [2]:
VINA_PATH = "vina"
RECEPTOR_PATH = Path("..") / "data" / "raw" / "3FXI.pdbqt"
LIGANDS_FOLDER = Path("..") / "data" / "raw" / "ligands"

DOCKING_FOLDER = Path("..") / "data" / "docking" / f"""run_{datetime.now().strftime("%Y-%m-%d")}"""
BEST_CSV_PATH = Path("..") / "data" / "processed" / "best_sepsis_candidates.csv"
LIGAND_PATH = Path("..") / "data" / "raw" / "curcumin.sdf"

DOCKING_OUTPUT_FOLDER = DOCKING_FOLDER / "output"
DOCKING_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

## Grid Parameters

Define grid box parameters (slightly smaller box than general screening might be sufficient for known ligand, but consistent center).

In [3]:
CENTER_X = -3
CENTER_Y = -9
CENTER_Z = -27
 
SIZE_X = 40
SIZE_Y = 40
SIZE_Z = 40

EXHAUSTIVENESS = 16 

## Prepare Curcumin

1.  Load from `.sdf`.
2.  Generate RDKit 3D Conformer + Optimize.
3.  Compute Gasteiger charges via OpenBabel.
4.  Save as `.pdbqt`.

In [4]:
mol = Chem.SDMolSupplier(str(LIGAND_PATH))[0]
mol_h = Chem.AddHs(mol)

params = AllChem.ETKDGv3()
params.useRandomCoords = True
embed_res = AllChem.EmbedMolecule(mol_h, params)

if embed_res == -1:
    AllChem.EmbedMolecule(mol_h)

AllChem.MMFFOptimizeMolecule(mol_h)

mol_block = Chem.MolToMolBlock(mol_h)
ob_mol = pybel.readstring("mol", mol_block)

ob_mol.calccharges(model="gasteiger")

output_path = LIGAND_PATH.with_suffix(".pdbqt")

ob_mol.write("pdbqt", str(output_path), overwrite=True)

## Run Docking

Dock Curcumin using Vina.

In [5]:
ligand_name = output_path.stem
output_file = DOCKING_OUTPUT_FOLDER / f"{ligand_name}_out.pdbqt"
log_file = DOCKING_FOLDER / f"{ligand_name}.log"

print(f"[1/1] Docking: {ligand_name}...")

command = [
    VINA_PATH,
    '--receptor', str(RECEPTOR_PATH),
    '--ligand', str(LIGAND_PATH.with_suffix(".pdbqt")),
    '--out', str(output_file),
    '--center_x', str(CENTER_X),
    '--center_y', str(CENTER_Y),
    '--center_z', str(CENTER_Z),
    '--size_x', str(SIZE_X),
    '--size_y', str(SIZE_Y),
    '--size_z', str(SIZE_Z),
    '--exhaustiveness', str(EXHAUSTIVENESS),
    '--cpu', '8'
]

result = {}
sub_result = subprocess.run(command, capture_output=True, text=True)

best_affinity = None

for line in sub_result.stdout.splitlines():
    if line.strip().startswith('1'):
        parts = line.split()
        if len(parts) >= 2:
            best_affinity = float(parts[1])
            break

if best_affinity is not None:
    result["Ligand"] = ligand_name
    result["Affinity"] = best_affinity
else:
    print(f"  Warning: Could not parse energy for {ligand_name}")
    with open(log_file, 'w') as f:
        f.write(sub_result.stderr)

print(result)

In [6]:
# Load comparison data from previous docking run
docking_scores_path = DOCKING_FOLDER / "final_docking_scores.csv"
df_scores = pd.read_csv(docking_scores_path)

## ADMET Prediction

Generate SMILES for Curcumin and run ADMET prediction.

In [7]:
# Get SMILES
smiles = Chem.MolToSmiles(mol)
print(f"SMILES: {smiles}")

# Predict ADMET
print("Running ADMET prediction...")
model = ADMETModel()
preds = model.predict([smiles])

## Calculate Sepsis Score

Apply the same weighting logic used in `04_admet_results.ipynb` to score Curcumin.

In [8]:
def scale_value(val, type='lower_better'):
    val = np.clip(val, 0, 1)
    if type == 'lower_better':
        return 1.0 - val
    return val

def normalize_log_s(val):
    # -2 (excellent), -6 (bad) -> scale to 0..1
    min_val, max_val = -6, -2
    norm = (val - min_val) / (max_val - min_val)
    return np.clip(norm, 0, 1)

# Weights
weights = {
    'hERG': 4.0,
    'AMES': 2.0,
    'DILI': 3.0,
    'Solubility': 3.5,
    'HIA_Hou': 0.0,
    'QED': 1.0,
    'BBB': 1.0
}

In [9]:
# Calculate scores for Curcumin
curcumin_row = preds.iloc[0].copy()

if 'Affinity' in result:
    curcumin_row['Affinity'] = result['Affinity']
else:
    curcumin_row['Affinity'] = None

curcumin_row['score_hERG'] = scale_value(curcumin_row['hERG'], 'lower_better') * 100
curcumin_row['score_AMES'] = scale_value(curcumin_row['AMES'], 'lower_better') * 100
curcumin_row['score_DILI'] = scale_value(curcumin_row['DILI'], 'lower_better') * 100
curcumin_row['score_Sol']  = normalize_log_s(curcumin_row['Solubility_AqSolDB']) * 100
curcumin_row['score_QED']  = scale_value(curcumin_row['QED'], 'higher_better') * 100
curcumin_row['score_BBB']  = scale_value(curcumin_row['BBB_Martins'], 'lower_better') * 100

# Weighted Sum
weighted_sum = (
    curcumin_row['score_hERG'] * weights['hERG'] +
    curcumin_row['score_AMES'] * weights['AMES'] +
    curcumin_row['score_DILI'] * weights['DILI'] +
    curcumin_row['score_Sol']  * weights['Solubility'] +
    curcumin_row['score_QED']  * weights['QED'] +
    curcumin_row['score_BBB']  * weights['BBB']
)

total_weight = sum(weights.values())
curcumin_row['Sepsis_Score_Norm'] = weighted_sum / total_weight

# Rounding
score_cols = ['score_hERG', 'score_AMES', 'score_DILI', 'score_Sol', 'score_QED', 'score_BBB', 'Sepsis_Score_Norm']
for col in score_cols:
    curcumin_row[col] = round(curcumin_row[col], 1)

raw_cols = ['hERG', 'Solubility_AqSolDB', 'DILI', 'AMES', 'QED', 'BBB_Martins']
for col in raw_cols:
    if col in curcumin_row:
        curcumin_row[col] = round(curcumin_row[col], 3)

print(f"Curcumin Sepsis Score: {curcumin_row['Sepsis_Score_Norm']:.2f}")

## Comparison

Compare Curcumin metrics with the top docked candidates.

In [10]:
cols_to_show = ['Ligand', 'Sepsis_Score_Norm', 'Affinity', 'score_hERG', 'score_AMES', 'score_DILI', 'score_Sol', 'score_QED', 'score_BBB']

curcumin_df = pd.DataFrame([curcumin_row])
curcumin_df['Ligand'] = 'Curcumin'

top_candidates = df_scores.head(5).copy()

comparison_df = pd.concat([curcumin_df, top_candidates], ignore_index=True)
existing_cols = [c for c in cols_to_show if c in comparison_df.columns]
comparison_df = comparison_df[existing_cols]

print("\n--- Comparison with Top Candidates ---")
print(comparison_df.to_string(index=False))

## Append Curcumin to Results

Add Curcumin to the results CSV file (`final_docking_scores.csv`).
> [!IMPORTANT]
> Performs type conversion on 'Affinity' column to ensure correct sorting.

In [11]:
print(f"Saving Curcumin data to {docking_scores_path}...")

if 'Curcumin' not in df_scores['Ligand'].values:
    common_cols = [c for c in df_scores.columns if c in curcumin_df.columns]
    curcumin_df_filtered = curcumin_df[common_cols]
    
    df_updated = pd.concat([df_scores, curcumin_df_filtered], ignore_index=True)
    
    # Ensure Affinity is numeric for correct sorting
    df_updated['Affinity'] = pd.to_numeric(df_updated['Affinity'], errors='coerce')
    
    df_updated = df_updated.sort_values(by='Affinity', ascending=True)
    
    df_updated.to_csv(docking_scores_path, index=False)
    print("Saved successfully.")
else:
    print("Curcumin already exists in the CSV. Skipping save.")